# Advanced Malicious URL Detection via LightGBM
**Project Domain:** Web Security / Network Defense  
**Target User:** Security Gateways, Browser Extensions & SOC Analysts

---

## 1. Abstract and Research Objective
The proliferation of phishing, malware distribution, and social engineering attacks often begins with a malicious URL. This notebook implements a high-performance detection system using **LightGBM** to classify URLs into safety categories based on their lexical and structural properties.

**Primary Objectives:**
* **Real-time Scalability:** Developing a classifier capable of rapid URL analysis at the network edge (e.g., Firewalls, Proxy).
* **Lexical Heuristics:** Evaluating the significance of string-based features (Length, Entropy, and Keyword frequency) to identify threats without the overhead of page crawling.
* **Proactive Defense:** Optimizing for high **Recall** to intercept malicious redirects before user exposure.

---

In [ ]:
import pandas as pd
import joblib
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, classification_report


# 1. Getting data
print("📂 Getting data for LightGBM...")
df = pd.read_csv("../data/processed_malicious_url.csv")

📂 Getting data for LightGBM...


## 2. Methodology: Feature Extraction & Numerical Normalization
Unlike file-based analysis, URL detection relies on **Static Lexical Analysis** extracted directly from the URI string. 

### 2.1. Feature Categories:
* **Structural Complexity**: URL length, hostname depth, and subdomain counts.
* **Statistical Entropy**: Measuring string randomness to detect **DGA (Domain Generation Algorithms)**.
* **Security Indicators**: Presence of raw IP addresses and high-risk TLDs.

### 2.2. Feature Scaling (Standardization)
As lexical features possess vastly different scales (e.g., URL length can be $>1000$ while Entropy is $<8$), we implement **StandardScaler**. This process transforms the data to have a mean ($\mu$) of 0 and a standard deviation ($\sigma$) of 1:
$$z = \frac{x - \mu}{\sigma}$$
* **Mathematical Rationale**: Normalization ensures that the gradient descent-based optimization in LightGBM converges faster and prevents features with large magnitudes from dominating the model's objective function.

---

In [3]:
X = df.drop(columns=['url', 'target'])
le = LabelEncoder()
y = le.fit_transform(df['target'])

# Standard Scaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

## 3. Experimental Setup
We utilize a **70/30 Hold-out Validation** strategy. This partitioning ensures that the model is evaluated on a substantial amount of unseen data, verifying its **Generalization** capability—the ability to detect new, previously unrecorded malicious URLs.

In [4]:
print("\n🚀 Splitting train test...")
x_train, x_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.3, stratify=y, random_state=42)


🚀 Splitting train test...


## 4. Model Configuration and Gradient Boosting Training
The core detection engine utilizes the **LightGBM Classifier**, a high-performance GBDT framework. For URL detection, we configure specific hyperparameters to capture complex lexical patterns while maintaining training efficiency.

### 4.1. Hyperparameter Rationale
* **`n_estimators=1500` & `learning_rate=0.05`**: We use a high number of boosting iterations combined with a small shrinkage rate to ensure the model converges steadily to a global minimum.
* **`num_leaves=128`**: This defines the tree complexity. A higher value allows the model to capture intricate interactions between URL features (e.g., specific character distributions in long paths).
* **`boosting_type='gbdt'`**: Standard Gradient Boosting Decision Tree, ideal for high-dimensional tabular data.
* **`importance_type='gain'`**: Configured to measure feature importance based on the **Total Gain** (contribution to loss reduction), providing a more accurate view for forensic analysis.
* **`force_row_wise=True`**: An optimization for memory efficiency when dealing with large-scale web traffic datasets.



### 4.2. Training with Early Stopping
To prevent **Overfitting**, we implement an automated validation strategy:
1. **`eval_set`**: The model continuously monitors performance on the unseen test set during training.
2. **`early_stopping(50)`**: Training will terminate if the validation loss does not improve for 50 consecutive rounds, ensuring the model generalizes well to new, "Zero-day" malicious URLs.

> **Technical Insight:** By combining a high `num_leaves` with `early_stopping`, we achieve a "Deep-but-Safe" model—powerful enough to detect subtle phishing heuristics without "memorizing" the training data.

In [5]:
# 4. Configure LightGBM
lgbm = lgb.LGBMClassifier(
    n_estimators=1500,
    learning_rate=0.05,
    num_leaves=128,
    boosting_type='gbdt',
    force_row_wise=True,
    importance_type='gain',
    n_jobs=-1,
    random_state=42
)

lgbm.fit(x_train, y_train, eval_set=[(x_test, y_test)], 
         callbacks=[lgb.early_stopping(stopping_rounds=50), lgb.log_evaluation(100)])


[LightGBM] [Info] Number of positive: 149126, number of negative: 303155
[LightGBM] [Info] Total Bins 1306
[LightGBM] [Info] Number of data points in the train set: 452281, number of used features: 14
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.329720 -> initscore=-0.709453
[LightGBM] [Info] Start training from score -0.709453
Training until validation scores don't improve for 50 rounds
[100]	valid_0's binary_logloss: 0.360773
[200]	valid_0's binary_logloss: 0.334824
[300]	valid_0's binary_logloss: 0.322433
[400]	valid_0's binary_logloss: 0.315692
[500]	valid_0's binary_logloss: 0.311104
[600]	valid_0's binary_logloss: 0.307595
[700]	valid_0's binary_logloss: 0.304811
[800]	valid_0's binary_logloss: 0.302501
[900]	valid_0's binary_logloss: 0.300076
[1000]	valid_0's binary_logloss: 0.298206
[1100]	valid_0's binary_logloss: 0.296571
[1200]	valid_0's binary_logloss: 0.295573
[1300]	valid_0's binary_logloss: 0.29465
[1400]	valid_0's binary_logloss: 0.293807
[1500]	valid_0's binary_lo

,boosting_type,'gbdt'
,num_leaves,128
,max_depth,-1
,learning_rate,0.05
,n_estimators,1500
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


## 5. Performance Metrics & Risk Assessment
In web security, we prioritize the balance between **Detection (Recall)** and **User Experience (Precision)**:

* **Recall (Sensitivity)**: The most critical metric; measures the success in capturing actual threats to prevent system breaches.
* **Precision**: Ensuring that benign URLs are not incorrectly blocked (minimizing false alarms).
* **F1-Score**: The harmonic mean providing a unified view of model reliability across diverse URL categories (Phishing, Malware, Defacement).
---

In [6]:
y_pred = lgbm.predict(x_test)
print(f"\n🎯 Accuracy LightGBM: {accuracy_score(y_test, y_pred)*100:.2f}%")
print(classification_report(y_test, y_pred, target_names=le.classes_))

c:\Users\vppho\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



🎯 Accuracy LightGBM: 87.03%
              precision    recall  f1-score   support

      benign       0.88      0.93      0.91    129925
        harm       0.85      0.74      0.79     63911

    accuracy                           0.87    193836
   macro avg       0.86      0.84      0.85    193836
weighted avg       0.87      0.87      0.87    193836



## 5. Model Persistence and Pipeline Integration
To transition from a research environment to a production-ready **URL Filtering Service**, we must ensure that the entire preprocessing and inference pipeline is preserved.

### 5.1. Artifact Serialization Strategy
We utilize the `joblib` library to serialize four critical components of the detection engine:
1. **`lgbm_model.pkl`**: The core Gradient Boosting engine containing the optimized decision boundaries.
2. **`scaler.pkl`**: The **StandardScaler** parameters ($\mu, \sigma$). This is mandatory to transform real-time user inputs into the same mathematical space as the training data.
3. **`label_encoder.pkl`**: Mapping for target classes (e.g., Phishing, Benign), ensuring the system returns human-readable security alerts.
4. **`feature_names.pkl`**: Preserving the exact order of features to prevent "Feature Mismatch" errors during inference.



### 5.2. Operational Readiness for Web Deployment
By saving these artifacts, the system achieves **Zero-training Inference**:
* **High-Speed Execution**: The Flask/Django backend can load these binary files instantaneously to analyze live web traffic.
* **Consistency**: Guarantees that the "Zero-day" detection logic remains identical across different environments (Development vs. Production).

> **Deployment Note:** The resulting model is now an autonomous unit, ready to be integrated into a **Real-time Gateway** or **Browser Security Extension** to intercept malicious redirects.

In [7]:
feature_names = X.columns.tolist()
joblib.dump(feature_names, "../models/feature_names.pkl")
joblib.dump(lgbm, "../models/lgbm_model.pkl")
joblib.dump(le, "../models/label_encoder.pkl")
joblib.dump(scaler, "../models/scaler.pkl")
print("💾 Đã lưu model và label encoder thành công!")

💾 Đã lưu model và label encoder thành công!


## 6. Conclusion and Future Directions
The LightGBM model provides a robust solution for automated URL filtering. 

**Summary of Findings:**
- Lexical features provide a strong baseline for identifying malicious intent without needing to visit the website.
- The high speed of LightGBM makes it suitable for integration into browser extensions or firewall systems.

**Future Work:**
- Integrating **Whois Data** (Domain age, Registrant info) for enhanced accuracy.
- Implementing **Ensemble techniques** to reduce false alarms.